# Proyecto 3 - Pruebas de aleatoriedad para `RandomLog`

Este notebook hace solo dos tipos de pruebas:

1. **Uniformidad**
2. **Independencia**

Archivo esperado: `RandomLog.csv`

Variables consideradas:
- `2`: número aleatorio usado para definir el tipo de tarea, esperado como uniforme continua `U(0,1)`.
- `3`: complejidad, esperada como uniforme discreta entre `1` y `10`.
- `4`, `5`, `6`: duraciones observadas en Desarrollo, QA y Retrabajo. A estas se les evalúa independencia, pero no uniformidad, porque no deberían ser uniformes.


In [11]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

ruta = Path("RandomLog.csv")

if not ruta.exists():
    raise FileNotFoundError("No se encontró RandomLog.csv. Sube el archivo al entorno del notebook.")

columnas = ["Tiempo", "Stream", "Variable", "Valor"]

def cargar_randomlog(ruta):
    intentos = [
        {"sep": ";", "decimal": ","},   # Excel español
        {"sep": ",", "decimal": "."},   # CSV estándar
        {"sep": "\t", "decimal": ","},
        {"sep": "\t", "decimal": "."},
    ]

    for cfg in intentos:
        try:
            df_temp = pd.read_csv(ruta, sep=cfg["sep"], decimal=cfg["decimal"], engine="python")
            if df_temp.shape[1] < 4:
                df_temp = pd.read_csv(ruta, sep=cfg["sep"], decimal=cfg["decimal"], header=None, engine="python")
            if df_temp.shape[1] >= 4:
                df_temp = df_temp.iloc[:, :4].copy()
                df_temp.columns = columnas
                for col in columnas:
                    df_temp[col] = pd.to_numeric(df_temp[col], errors="coerce")
                df_temp = df_temp.dropna(subset=columnas).copy()
                df_temp["Stream"] = df_temp["Stream"].astype(int)
                df_temp["Variable"] = df_temp["Variable"].astype(int)
                return df_temp
        except Exception:
            pass

    raise ValueError("No se pudo leer el CSV. Revisa separador, decimales o formato del archivo.")

df = cargar_randomlog(ruta)

nombres = {
    2: "Aleatorio tipo de tarea",
    3: "Complejidad",
    4: "Duración Desarrollo",
    5: "Duración QA y Testing",
    6: "Duración Retrabajo"
}

df["NombreVariable"] = df["Variable"].map(nombres).fillna("Variable no identificada")

print("Archivo cargado correctamente.")
print("Registros:", len(df))
display(df.head())

Archivo cargado correctamente.
Registros: 2623


,Tiempo,Stream,Variable,Valor,NombreVariable
0,0.00,0,2,0.33,Aleatorio tipo de tarea
1,0.00,0,3,3.00,Complejidad
2,69.04,0,2,0.22,Aleatorio tipo de tarea
3,69.04,0,3,6.00,Complejidad
4,148.64,0,2,0.63,Aleatorio tipo de tarea


## 1. Pruebas de uniformidad

Se aplican solo donde tiene sentido:

- Variable `2`: prueba KS contra `U(0,1)`.
- Variable `3`: prueba chi-cuadrado contra uniforme discreta `{1,2,...,10}`.

Las duraciones observadas (`4`, `5`, `6`) no se prueban contra uniforme porque no fueron modeladas como uniformes.


In [12]:
resultados_uniformidad = []

# Variable 2: uniforme continua U(0,1)
x = df.loc[df["Variable"] == 2, "Valor"].dropna()

if len(x) >= 20:
    ks_stat, p_value = stats.kstest(x, "uniform", args=(0, 1))
    resultados_uniformidad.append({
        "Variable": 2,
        "Prueba": "KS contra U(0,1)",
        "n": len(x),
        "Estadístico": ks_stat,
        "p_value": p_value,
        "Conclusión": "No se rechaza uniformidad" if p_value >= 0.05 else "Se rechaza uniformidad"
    })
else:
    resultados_uniformidad.append({
        "Variable": 2,
        "Prueba": "KS contra U(0,1)",
        "n": len(x),
        "Estadístico": np.nan,
        "p_value": np.nan,
        "Conclusión": "Muestra insuficiente"
    })

# Variable 3: uniforme discreta 1..10
x = df.loc[df["Variable"] == 3, "Valor"].dropna().round().astype(int)

if len(x) >= 20:
    observadas = x.value_counts().reindex(range(1, 11), fill_value=0).values
    esperadas = np.ones(10) * (observadas.sum() / 10)

    if np.all(esperadas >= 5):
        chi2_stat, p_value = stats.chisquare(f_obs=observadas, f_exp=esperadas)
        conclusion = "No se rechaza uniformidad discreta" if p_value >= 0.05 else "Se rechaza uniformidad discreta"
    else:
        chi2_stat, p_value = np.nan, np.nan
        conclusion = "Frecuencias esperadas insuficientes"

    resultados_uniformidad.append({
        "Variable": 3,
        "Prueba": "Chi-cuadrado contra uniforme discreta 1..10",
        "n": len(x),
        "Estadístico": chi2_stat,
        "p_value": p_value,
        "Conclusión": conclusion
    })
else:
    resultados_uniformidad.append({
        "Variable": 3,
        "Prueba": "Chi-cuadrado contra uniforme discreta 1..10",
        "n": len(x),
        "Estadístico": np.nan,
        "p_value": np.nan,
        "Conclusión": "Muestra insuficiente"
    })

uniformidad = pd.DataFrame(resultados_uniformidad)
display(uniformidad)

,Variable,Prueba,n,Estadístico,p_value,Conclusión
0,2,"KS contra U(0,1)",510,0.024902,0.901884,No se rechaza uniformidad
1,3,Chi-cuadrado contra uniforme discreta 1..10,510,7.960784,0.538103,No se rechaza uniformidad discreta


## 2. Pruebas de independencia

Se aplican dos revisiones por variable:

1. **Prueba de rachas respecto a la mediana**: detecta patrones en la secuencia.
2. **Autocorrelación lag 1**: mide relación lineal entre cada valor y el siguiente.

Criterio general:
- `p_value >= 0.05`: no hay evidencia fuerte contra independencia.
- `p_value < 0.05`: posible dependencia o patrón.


In [13]:
def prueba_rachas_mediana(valores):
    """
    Prueba de rachas respecto a la mediana.

    H0: la secuencia no presenta evidencia fuerte de dependencia o patrón.
    H1: la secuencia presenta posible dependencia o patrón.

    Si p_value >= 0.05: no se rechaza independencia.
    Si p_value < 0.05: posible dependencia.
    """

    x = np.asarray(valores, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 10:
        return {
            "n": len(x),
            "rachas": np.nan,
            "z": np.nan,
            "p_value": np.nan,
            "conclusion": "Muestra insuficiente"
        }

    mediana = np.median(x)

    # Se eliminan valores iguales a la mediana para evitar ambigüedad
    x = x[x != mediana]

    if len(x) < 10:
        return {
            "n": len(x),
            "rachas": np.nan,
            "z": np.nan,
            "p_value": np.nan,
            "conclusion": "Muestra insuficiente después de eliminar empates con la mediana"
        }

    signos = x > mediana

    n1 = np.sum(signos)
    n2 = len(signos) - n1

    if n1 == 0 or n2 == 0:
        return {
            "n": len(x),
            "rachas": np.nan,
            "z": np.nan,
            "p_value": np.nan,
            "conclusion": "No hay dos grupos alrededor de la mediana"
        }

    rachas = 1 + np.sum(signos[1:] != signos[:-1])

    n = n1 + n2
    media = 1 + (2 * n1 * n2) / n
    varianza = (2 * n1 * n2 * (2 * n1 * n2 - n)) / ((n ** 2) * (n - 1))

    z = (rachas - media) / np.sqrt(varianza)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    if p_value >= 0.05:
        conclusion = "No se rechaza independencia"
    else:
        conclusion = "Posible dependencia o patrón"

    return {
        "n": n,
        "rachas": rachas,
        "z": z,
        "p_value": p_value,
        "conclusion": conclusion
    }


# ============================================================
# APLICAR LA PRUEBA POR VARIABLE
# ============================================================

resultados_rachas = []

for variable, datos in df.groupby("Variable"):
    datos_ordenados = datos.sort_values("Tiempo")
    valores = datos_ordenados["Valor"].dropna().values

    resultado = prueba_rachas_mediana(valores)

    resultados_rachas.append({
        "Variable": variable,
        "Nombre": nombres.get(variable, f"Variable {variable}"),
        "n": resultado["n"],
        "Rachas": resultado["rachas"],
        "z": resultado["z"],
        "p_value": resultado["p_value"],
        "Conclusion": resultado["conclusion"]
    })

rachas = pd.DataFrame(resultados_rachas)

display(rachas)

,Variable,Nombre,n,Rachas,z,p_value,Conclusion
0,2,Aleatorio tipo de tarea,504,246,-0.623886,0.532702,No se rechaza independencia
1,3,Complejidad,465,259,2.381037,0.017264,Posible dependencia o patrón
2,4,Duración Desarrollo,548,275,0.000000,1.000000,No se rechaza independencia
3,5,Duración QA y Testing,754,361,-1.239030,0.215334,No se rechaza independencia
4,6,Duración Retrabajo,300,159,0.925309,0.354805,No se rechaza independencia


## 3. Conclusión automática

In [14]:
print("CONCLUSIÓN GENERAL\n")

for _, row in uniformidad.iterrows():
    print(
        f"Uniformidad - Variable {int(row['Variable'])}: "
        f"{row['Conclusión']} | p-value = {row['p_value']}"
    )

print()

for _, row in rachas.iterrows():
    print(
        f"Independencia - Variable {int(row['Variable'])} ({row['Nombre']}): "
        f"{row['Conclusion']} | p-value rachas = {row['p_value']}"
    )

print(
    "\nNota: las duraciones observadas se usan para independencia, "
    "no para uniformidad, porque no fueron modeladas como variables uniformes."
)

CONCLUSIÓN GENERAL

Uniformidad - Variable 2: No se rechaza uniformidad | p-value = 0.90188425086405
Uniformidad - Variable 3: No se rechaza uniformidad discreta | p-value = 0.538103014543577

Independencia - Variable 2 (Aleatorio tipo de tarea): No se rechaza independencia | p-value rachas = 0.5327022237596712
Independencia - Variable 3 (Complejidad): Posible dependencia o patrón | p-value rachas = 0.017263997501544726
Independencia - Variable 4 (Duración Desarrollo): No se rechaza independencia | p-value rachas = 1.0
Independencia - Variable 5 (Duración QA y Testing): No se rechaza independencia | p-value rachas = 0.21533439877731975
Independencia - Variable 6 (Duración Retrabajo): No se rechaza independencia | p-value rachas = 0.35480516414127106

Nota: las duraciones observadas se usan para independencia, no para uniformidad, porque no fueron modeladas como variables uniformes.


In [15]:
# Guardar resultados en Excel
salida = "resultados_pruebas_aleatoriedad.xlsx"

with pd.ExcelWriter(salida) as writer:
    df.to_excel(writer, sheet_name="Datos_limpios", index=False)
    uniformidad.to_excel(writer, sheet_name="Uniformidad", index=False)
    independencia.to_excel(writer, sheet_name="Independencia", index=False)

print(f"Archivo generado: {salida}")

Archivo generado: resultados_pruebas_aleatoriedad.xlsx
